# Crawl Test — Facebook & Threads

Self-contained notebook to develop and test the social media connectors.
No need to clone the repo.

**Required secrets** (Runtime → Secrets 🔑):
- `FB_PAGE_IDS` — comma-separated list of page IDs to crawl, e.g. `111111,222222,333333`
- `FB_ACCESS_TOKEN`
- `THREADS_ACCESS_TOKEN`

## Setup

In [ ]:
!pip install requests -q

In [ ]:
import os
from google.colab import userdata

# Comma-separated page IDs — crawls every page in the list
_raw_ids        = userdata.get("FB_PAGE_IDS")
FB_PAGE_IDS     = [pid.strip() for pid in _raw_ids.split(",") if pid.strip()]
FB_ACCESS_TOKEN = userdata.get("FB_ACCESS_TOKEN")
THREADS_TOKEN   = userdata.get("THREADS_ACCESS_TOKEN")

# How many days back to fetch
DAYS_BACK = 1

# Keywords to filter by (Vietnamese + English)
KEYWORDS = [
    "zalopay",
    "ví zalopay",
    "nạp tiền lỗi",
    "thanh toán lỗi",
    "zalo pay",
]

print(f"FB_PAGE_IDS    : {'✅ ' + str(FB_PAGE_IDS) if FB_PAGE_IDS else '❌ missing'}")
print(f"FB_ACCESS_TOKEN: {'✅' if FB_ACCESS_TOKEN else '❌ missing'}")
print(f"THREADS_TOKEN  : {'✅' if THREADS_TOKEN else '❌ missing'}")

In [ ]:
import requests
from datetime import datetime, timedelta, timezone


def _paginate(url: str, params: dict, max_pages: int = 10) -> list[dict]:
    results = []
    for _ in range(max_pages):
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        body = resp.json()
        results.extend(body.get("data", []))
        paging   = body.get("paging", {})
        next_url = paging.get("next")
        after    = paging.get("cursors", {}).get("after")
        if next_url:
            url, params = next_url, {}
        elif after:
            params = {**params, "after": after}
        else:
            break
    return results


def _matches(text: str) -> bool:
    lower = text.lower()
    return any(kw.lower() in lower for kw in KEYWORDS)


print("Shared helpers defined ✅")

---
## Facebook Connector

Crawls every page in `FB_PAGE_IDS`. For each page it fetches:
- Posts on the page timeline (by the page and by visitors)
- Comments on each post

Only items matching `KEYWORDS` are kept.

In [ ]:
_FB_BASE = "https://graph.facebook.com/v20.0"


def fb_fetch() -> list[dict]:
    """Fetch posts + comments from all pages in FB_PAGE_IDS."""
    since_ts = int((datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)).timestamp())
    items = []
    for page_id in FB_PAGE_IDS:
        print(f"  Fetching page {page_id}...")
        items.extend(_fetch_one_page(page_id, since_ts))
    return items


def _fetch_one_page(page_id: str, since_ts: int) -> list[dict]:
    items = []
    for post in _fb_get_feed(page_id, since_ts):
        text = post.get("message") or post.get("story") or ""
        if text and _matches(text):
            items.append(_fb_make_item(post["id"], text, _fb_images(post), post["created_time"]))

        for comment in _fb_get_comments(post["id"]):
            ctext = comment.get("message", "")
            if ctext and _matches(ctext):
                items.append(_fb_make_item(comment["id"], ctext, [], comment["created_time"]))
    return items


def _fb_get_feed(page_id: str, since_ts: int) -> list[dict]:
    params = {
        "access_token": FB_ACCESS_TOKEN,
        "fields": "id,message,story,created_time,attachments{media{image{src}},type}",
        "since": since_ts,
        "limit": 100,
    }
    return _paginate(f"{_FB_BASE}/{page_id}/feed", params)


def _fb_get_comments(post_id: str) -> list[dict]:
    params = {
        "access_token": FB_ACCESS_TOKEN,
        "fields": "id,message,created_time",
        "limit": 100,
    }
    return _paginate(f"{_FB_BASE}/{post_id}/comments", params)


def _fb_images(post: dict) -> list[str]:
    urls = []
    for att in (post.get("attachments") or {}).get("data", []):
        src = ((att.get("media") or {}).get("image") or {}).get("src")
        if src:
            urls.append(src)
    return urls


def _fb_make_item(item_id, text, images, timestamp) -> dict:
    return {"id": item_id, "source": "facebook", "text": text, "images": images, "timestamp": timestamp}


print("Facebook connector defined ✅")

In [ ]:
fb_items = fb_fetch()

print(f"\nTotal: {len(fb_items)} Facebook items\n")
for item in fb_items:
    print(f"  [{item['id']}] {item['timestamp']}")
    print(f"  {item['text'][:120]}")
    if item['images']:
        print(f"  images: {item['images']}")
    print()

---
## Threads Connector

Fetches threads and replies from the authenticated Zalopay Threads account.
Only items newer than `DAYS_BACK` days and matching `KEYWORDS` are kept.

In [ ]:
_TH_BASE = "https://graph.threads.net/v1.0"


def th_fetch() -> list[dict]:
    """Fetch threads + replies from the Zalopay Threads account."""
    cutoff = datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)
    items = []

    for thread in _th_get_threads():
        if _older_than(thread.get("timestamp", ""), cutoff):
            continue
        text = thread.get("text", "")
        if text and _matches(text):
            items.append(_th_make_item(thread["id"], text, _th_images(thread), thread.get("timestamp", "")))

        for reply in _th_get_replies(thread["id"]):
            if _older_than(reply.get("timestamp", ""), cutoff):
                continue
            rtext = reply.get("text", "")
            if rtext and _matches(rtext):
                items.append(_th_make_item(reply["id"], rtext, _th_images(reply), reply.get("timestamp", "")))

    return items


def _th_get_threads() -> list[dict]:
    params = {
        "access_token": THREADS_TOKEN,
        "fields": "id,text,timestamp,media_type,media_url,thumbnail_url",
        "limit": 100,
    }
    return _paginate(f"{_TH_BASE}/me/threads", params)


def _th_get_replies(thread_id: str) -> list[dict]:
    params = {
        "access_token": THREADS_TOKEN,
        "fields": "id,text,timestamp,media_type,media_url",
        "limit": 100,
    }
    return _paginate(f"{_TH_BASE}/{thread_id}/replies", params)


def _th_images(item: dict) -> list[str]:
    if item.get("media_type") in ("IMAGE", "VIDEO"):
        url = item.get("media_url") or item.get("thumbnail_url")
        if url:
            return [url]
    return []


def _older_than(ts_str: str, cutoff: datetime) -> bool:
    if not ts_str:
        return False
    try:
        ts = datetime.fromisoformat(ts_str.replace("Z", "+00:00"))
        return ts < cutoff
    except ValueError:
        return False


def _th_make_item(item_id, text, images, timestamp) -> dict:
    return {"id": item_id, "source": "threads", "text": text, "images": images, "timestamp": timestamp}


print("Threads connector defined ✅")

In [ ]:
th_items = th_fetch()

print(f"Total: {len(th_items)} Threads items\n")
for item in th_items:
    print(f"  [{item['id']}] {item['timestamp']}")
    print(f"  {item['text'][:120]}")
    if item['images']:
        print(f"  images: {item['images']}")
    print()

---
## Combined result

In [ ]:
all_items = fb_items + th_items
print(f"Total: {len(all_items)} items  ({len(fb_items)} Facebook + {len(th_items)} Threads)")